# Introduction

In this notebook, I'll be demonstrating unsupervised clustering and dimensionality reduction. 

If you haven't already, please refer to [01-data-exploration.ipynb](), as that notebook describes most of the data loading and pre-processing steps that we'll perform at the beginning of this notebook.

## Imports

In [ ]:
import os
import pandas as pd
import numpy as np

# We use two different plotting libraries, depending on which kind of plot we want
import matplotlib.pyplot as plt
import seaborn as sns

# SKLearn Imports
from sklearn import metrics
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


from data_loading_functions import load_data, draw_conf_mat

# Turn off warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Turn on Latex support for matplotlib (if you want it)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Bookman"
})

# Plotting values
background_color = "#f7f7f7"
plot_colors = ['#ef8a62', '#67a9cf', '#a6dba0']

# Set an option for Pandas to display smaller floating-point numbers
pd.options.display.float_format = '{:,.2f}'.format

# Machine Epsilon, for computing division
eps = np.finfo(np.float32).eps

## Data Loading

In [ ]:
data = load_data(data_loc='../data/wisconsin_breast_cancer_data.csv')

# Split apart the pieces of the `data` dictionary
training_values = data['training_values']
training_labels = data['training_labels']
testing_values = data['testing_values']
testing_labels = data['testing_labels']

In [ ]:
feature_names = list(training_values.columns)

# Clustering

Clustering is **slightly** different than in other classification approaches: Here, you aren't transforming the data, but you are assigning a class label to each input data point. So here, when you run the `.fit()` method, you're done -- you can pull out the assigned labels from the fit model, no need for an additional `.transform()` or `.predict()` step.

Note also that you should set the number of clusters to use when doing K-Means; if you don't provide a number of desired output clusters, it defaults to 8.

In [ ]:
help(KMeans)

## K-Means Clustering (k=2)

In [ ]:
# Set the number of clusters
n_clusters = 2

# Instantiate the model -- use the "verbose" tag to see what the algo is doing
kmeans = KMeans(n_clusters=n_clusters,
                n_init=4,
                random_state=0,
                verbose=True)

# Fit the model to the training data
kmeans.fit(training_values)

# Extract the cluster memberships (on training data)
y_clustering = kmeans.labels_

In [ ]:
# You can pull out the centroids of each cluster
# Note these are d-dimensional vectors!
kmeans.cluster_centers_

In [ ]:
# y_clustering contains the cluster label for the data point
# Note that you cannot simply match y_clustering to the training_label, because a cluster of 0 may not line up with a class label of 0!
y_clustering

## Visualize

Now we will use PCA to get the low-dimensional representation of the data; however, we will only use this to visualize the clusters. Recall that there are many ways to do low-dimensional embeddings, and this may not give us the "best" view of the dataset.

Here we compare the cluster labels (obtained in an unsupervised fashion) with the known class lables. In most cases, you **won't be able to do this** because remember, you don't typically have the true class labels.

In [ ]:
Training_PCA = PCA(n_components=2).fit(training_values)

X_train_reduced = Training_PCA.transform(training_values)
X_centroids = Training_PCA.transform(kmeans.cluster_centers_)

In [ ]:
# Plot the data
f, ax = plt.subplots(figsize=(10,6), facecolor=background_color)

ax.scatter(X_train_reduced[y_clustering==0,0], X_train_reduced[y_clustering==0,1], alpha=.8, label="Cluster 0")
ax.scatter(X_train_reduced[y_clustering==1,0], X_train_reduced[y_clustering==1,1], alpha=.8, label="Cluster 1")
ax.scatter(X_centroids[:,0], X_centroids[:,1], c='k', s=50, label="Cluster Centroids")

# Annotate Plot
ax.set(xlabel=r'$x_{1}$',
       ylabel=r'$x_{2}$',
       title=f'K-Means Cluster Plot (k={n_clusters})',
       facecolor=background_color)

ax.legend(frameon=True)
ax.grid(linestyle=':')
plt.tight_layout()


plt.show()

In [ ]:
# Plot the data itself
X_train_reduced = PCA(n_components=2).fit_transform(training_values)

fig, ax = plt.subplots(1,2,figsize=(20,6), facecolor=background_color)

ax[0].scatter(X_train_reduced[y_clustering==0,0], X_train_reduced[y_clustering==0,1], alpha=.8, label="Cluster 0")
ax[0].scatter(X_train_reduced[y_clustering==1,0], X_train_reduced[y_clustering==1,1], alpha=.8, label="Cluster 1")
ax[0].set(xlabel=r'$x_{1}$',
          ylabel=r'$x_{2}$',
          title=f'K-Means Cluster Plot (k={n_clusters})',
          facecolor=background_color)

ax[1].scatter(X_train_reduced[training_labels['diagnosis_label']==0,0], X_train_reduced[training_labels['diagnosis_label']==0,1], alpha=.8, label="Class 0")
ax[1].scatter(X_train_reduced[training_labels['diagnosis_label']==1,0], X_train_reduced[training_labels['diagnosis_label']==1,1], alpha=.8, label="Class 1")
ax[1].set(xlabel=r'$x_{1}$',
          ylabel=r'$x_{2}$',
          title=f'Actual Class Memberships',
          facecolor=background_color)

for a in ax.ravel():
    a.legend(frameon=True)
    a.grid(linestyle=':')
plt.tight_layout()

plt.show()

## Evaluation

We discussed some metrics in the class. However, there are three we will go over here because they are straightforward to extract.

### Silhouette Coefficient

[User Guide](https://scikit-learn.org/stable/modules/clustering.html#silhouette-coefficient)

The Silhouette Coefficient calculates two values: 

- $a$, The mean distance between a sample and all other points in the *same* cluster
- $b$: The mean distance between a sample and all other points in the *next nearest* cluster.

If our clustering is good, we want $a$ to be **low** (clusters are tightly packed) and $b$ to be **high** (different clusters are far apart).

The Silhouette Coefficient for a single point is given as:

$$s = \frac{b-a}{max(a, b)}$$

The overall Silhouette Coefficient is the mean $s$ for all data points. Thus, a good clustering will have a **high Silhouette Coefficient**.

### Calinski-Harabasz Index

[User Guide](https://scikit-learn.org/stable/modules/clustering.html#calinski-harabasz-index)

The index is the ratio of the sum of **between-clusters dispersion** and of **within-cluster dispersion** for all clusters (where dispersion is defined as the sum of distances squared).

If clusters are tight and far apart, then the between-cluster dispersion should be high and the within-cluster dispersion should be low. Thus, a good clustering will have a **high Calinski-Harabasz Index**.


### Davies-Bouldin Index

This metric calculates the average similarity between each cluster and the "most-similar" other cluster. Similarity, in this case, includes $s_{i}$, the distance between each point of cluster $i$ and the centroid of that cluster (so "larger" clusters will have a higher $s_{i}$) and $d_{ij}$ which is the distance between cluster centroids $i$ and $j$.

The similarity between two clusters is defined as:

$$ R_{ij} = \frac{s_{i}+s_{j}}{d_{ij}} $$

The Davies-Bouldin Index is then:

$$ DB = \frac{1}{k} \sum_{i=1}^{k}\textrm{max}_{i\neq j}R_{ij}$$

In a good clustering, the similarity $R_{ij}$ between two different clusters $i$ and $j$ should be low. Thus, a good clustering will have a **low Davies-Bouldin Index**.

In [ ]:
# Calculate our metrics
eval_metrics = [metrics.silhouette_score,
                metrics.calinski_harabasz_score,
                metrics.davies_bouldin_score]

scores = [m(training_values, y_clustering) for m in eval_metrics]

silhouette_scores = [scores[0]]
ch_scores = [scores[1]]
db_scores = [scores[2]]

print(f'k={n_clusters}')
print(f'Silhouette Score\tCalinski Harabasz\tDavies Bouldin')
print(f'(Higher is better)\t(Higher is better)\t(Lower is better)')

print(f'\t'.join([str(x) for x in scores]))

In [ ]:
# What if we try a different cluster number?
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, n_init=5, random_state=0, verbose=False)
kmeans.fit(training_values)
y_clustering = kmeans.labels_

In [ ]:
X_centroids = Training_PCA.transform(kmeans.cluster_centers_)

f, ax = plt.subplots(figsize=(10,6), dpi=150, facecolor=background_color)

for c_idx in range(n_clusters):
    ax.scatter(X_train_reduced[y_clustering==c_idx,0], X_train_reduced[y_clustering==c_idx,1], alpha=.8, label=f"Cluster {c_idx}")

ax.scatter(X_centroids[:,0], X_centroids[:,1], c='k', marker='x', s=50, label="Cluster Centroid")

# Annotate Plot
ax.set(xlabel=r'$x_{1}$',
       ylabel=r'$x_{2}$',
       title=f'K-Means Cluster Plot (k={n_clusters})',
       facecolor=background_color)

ax.legend(frameon=True)
ax.grid(linestyle=':')
plt.tight_layout()

plt.show()

In [ ]:
scores = [m(training_values, y_clustering) for m in eval_metrics]

print(f'k={n_clusters}')
print(f'Silhouette Score\tCalinski Harabasz\tDavies Bouldin')
print(f'(Higher is better)\t(Higher is better)\t(Lower is better)')

print(f'\t'.join([str(x) for x in scores]))

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(18,6), facecolor=background_color)

ax[0].scatter(X_train_reduced[:,0], X_train_reduced[:,1], c=y_clustering)
ax[0].set_title(f'K-means with k={n_clusters}')

ax[1].scatter(X_train_reduced[:,0], X_train_reduced[:,1], c=training_labels['diagnosis_label'])
ax[1].set_title(f'Actual Class Memberships')

for c_idx in range(n_clusters):
    ax[0].scatter(X_train_reduced[y_clustering==c_idx,0], X_train_reduced[y_clustering==c_idx,1], alpha=.8, label=f'Cluster {c_idx}')

ax[0].set(xlabel=r'$x_{1}$',
          ylabel=r'$x_{2}$',
          title=f'K-Means Cluster Plot (k={n_clusters})',
          facecolor=background_color)

ax[1].scatter(X_train_reduced[training_labels['diagnosis_label']==0,0], X_train_reduced[training_labels['diagnosis_label']==0,1], alpha=.8, label="Class 0")
ax[1].scatter(X_train_reduced[training_labels['diagnosis_label']==1,0], X_train_reduced[training_labels['diagnosis_label']==1,1], alpha=.8, label="Class 1")
ax[1].set(xlabel=r'$x_{1}$',
          ylabel=r'$x_{2}$',
          title=f'Actual Class Memberships',
          facecolor=background_color)

for a in ax.ravel():
    a.legend(frameon=True)
    a.grid(linestyle=':')

plt.tight_layout()
plt.show()

In [ ]:
cluster_metrics = {
    'Silhouette Scores': silhouette_scores,
    'CB': ch_scores,
    'DB': db_scores}

In [ ]:
# Compute the cluster scores for different n_clusters
sil_scores = dict()
for n_clusters in [2, 3, 4, 5, 10, 100]:
    kmeans = KMeans(n_clusters=n_clusters, n_init=5, random_state=0, verbose=False)
    kmeans.fit(training_values)
    y_clustering = kmeans.labels_
    sil_scores[n_clusters] = metrics.silhouette_score(training_values, y_clustering)

In [ ]:
sil_scores

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=150, facecolor=background_color)

width = 0.75

x = np.arange(len(sil_scores.keys()))
multiplier = 0

ax.bar(range(len(sil_scores.keys())), sil_scores.values(), width, label='Silhouette', color=plot_colors[0], edgecolor='k')
ax.grid(which='major', axis='y', zorder=0, color='gray', linestyle=':', dashes=(1,5))

# # Annotate Plot
# ax.set(xlabel=r'Hidden Layer Architecture',
#        ylabel=r'Accuracy',
#        ylim=(0.5, 1.0))

ax.set_xticks(x, [f'{x} clusters' for x in sil_scores.keys()])

# Line up the title with the left side of the plot
text_x = ax.get_xlim()[0]
text_y = ax.get_ylim()[1] + 0.01

ax.text(text_x, text_y+0.02, 'Silhouette Scores for K-Means Clustering', fontsize=16, fontweight='bold', fontfamily='serif')
ax.text(text_x, text_y, 'Higher scores indicate better clustering', fontsize=12, fontweight='light', fontfamily='serif')

plt.tight_layout()
plt.show()

# Pipelines

Setting up a pipeline means we can train and apply several steps at once, including feature reduction / selection (here we are doing PCA) and classification. 
Normally we might also do feature scaling here as well, but scaling is currently performed inside the data loading script.

To set up the pipeline we first need to define all the parameters for each of the components. 
This means the number of components for PCA and the characteristics of the MLP.

In [ ]:
# PCA Parameters
n_components = 2

To create a pipeline, you pass in a list of "steps". Each step is represented by a tuple, where the first value is the name of the step (a string) and the second value is the initialized object that implements this step.

The steps of a pipeline can be any `sklearn` class that is a `transformer`, which means a class with a `.transform()` and/or a `.fit_transform()` method.

After construction, you can call a `.fit()`, `.transform()`, or `.fit_transform()` on the pipeline object and it automatically runs all steps of the process. 

From the documentation, the benefits of this are:

- *Convenience* and *Encapsulation*
- *Joint Parameter Selection* (more on this below)
- *Safety*

More details are given here: https://scikit-learn.org/stable/modules/compose.html

In [ ]:
# Set up your pipeline
classifier = Pipeline([
    ('feature_reduction', PCA(n_components=n_components)),
    ('classifier', DecisionTreeClassifier()) # DEFAULTS
])

In [ ]:
# Train the pipeline -- trains each step with the training data passed in here
classifier.fit(training_values, training_labels)

In [ ]:
# Treat the whole pipeline as a single object
testing_predicted = classifier.predict(testing_values)
testing_accuracy = classifier.score(testing_values, testing_labels)
  
print(f"MLP Accuracy on the Testing Set: {testing_accuracy * 100:0.5}%")

# Hyperparameter Tuning (Automatic)

By searching manually, we can optimize a few parameters. What if we want to optimize *many* parameters and just pick the optimal result?
This is where **grid search** comes into play: Define all the parameters and the range of values you want to test, and let the program find the best combination.

To perform automatic hyperparameter searches, the easiest approach is to use a `GridSearchCV` object.
This will automatically try out each combination of parameter values that you specify for the input estimator, and report the best performance.
"Performance" is defined by whatever scoring function is used by the estimator your provide, or you can develop your own.

In this example, our estimator is the pipeline object that defines the feature reduction and classification steps (the data is already scaled, and the `StandardScaler()` doesn't have any tunable parameters anyway).
In this way we are not only optimizing the classifier, but also selecting the proper feature reduction and selection parameters too!

In [ ]:
cv_classifier = Pipeline([
                          ('feature_reduction', PCA()),
                          ('classifier', DecisionTreeClassifier())
                       ])

A `parameter grid` allows you to specify all of the hyperparameters you want to try out.

The grid object is a `dict()` where:
- Keys are named by the step of the pipeline, followed by two underscroes, followed by the named parameter; and 
- Values are lists of the parameter values to try out

You then pass this to `GridSearchCV`, along with the "thing" you want to optimize (in our case, the pipeline) and other parameters dictating how the object should be trained, like `n_jobs`.

In [ ]:
param_grid = {
    'feature_reduction__n_components': [2, 4, 10, 20, 30],
    'classifier__criterion': ['gini', 'entropy', 'log_loss'],
    'classifier__max_depth': [2, 5, 10],
    }

grid = GridSearchCV(cv_classifier, n_jobs=-1, param_grid=param_grid)

In [ ]:
# Display the "complexity" of the space
grid_complexity = 1
for k,v in param_grid.items():
    grid_complexity *= len(v)

print(f"Testing a total of {grid_complexity} parameter combinations.")

In [ ]:
# WARNING: This might take awhile!
grid.fit(training_values, training_labels)

We can view the parameter values that yielded the best performance:

In [ ]:
grid.best_params_

As well as the best score (in this case, accuracy):

In [ ]:
grid.best_score_

Finally, we can call the `.predict()` method on the trained grid object to run prediction using just the best estimator that it found:

In [ ]:
# X_test_predicted = grid.predict(X_test)
testing_accuracy = grid.score(testing_values, testing_labels)

print(f"Optimized Accuracy on the Testing Set: {100*testing_accuracy:0.5}%")